# 语音（Audio）

> 声音是振荡的波，计算机用数字去逼近它

语音任务有四类：

| 任务 | 英文 | 典型应用 |
|------|------|----------|
| 音频分类 | Audio Classification | 语音指令识别、环境声音分类、音乐流派分类 |
| 音频到音频 | Audio to Audio | 语音增强、降噪、源分离、超分辨率 |
| 语音识别 | Automatic Speech Recognition (ASR) | 语音转文字、实时字幕 |
| 语音合成 | Text to Speech (TTS) | 语音助手、有声书、配音 |

---

## 1. 格式：声音是如何存到计算机中的？

图像很简单，由像素点构成。一张 1920×1080 的图就是宽 1920 像素、高 1080 像素。

简单举例：
```
X  X  X  X  X
X  X  X  X  X
X  X  X  X  X
X  X  X  X  X
X  X  X  X  X
```

每一个 X 可以是一个数字（比如 0/1），就很显然是个 5×5 的黑白图。

X 还可以是一个表示三基色的数组（比如 `(0, 255, 0)`），这是一个绿色的像素点，整个图便是彩色图。

复习了图像的格式，那么音频呢？似乎不太好办。

### 1.1 声音的本质

回到初中的物理与生物课上：

在物理课上，提到过声音：一种物体振荡的波，通过介质传播，经过人耳捕获、大脑处理，变为声音。

声音有这么几个特征：
- **音调**（高低）：由振荡的**频率**决定
- **响度**（大小）：由振荡的**幅度**决定
- **音色**：由**波形**决定

声音的本质是振荡波——它随时间连续变化。计算机无法存储连续信号，所以需要**采样（Sampling）**。

### 1.2 采样与量化

**采样**：每隔一小段时间取一个值，把连续波变成离散序列。

```
连续波：    ～～～～～～～～～～～～～～
                     ↓ 采样
离散序列：  [0.1, 0.3, 0.7, 0.9, 0.6, 0.2, -0.1, -0.4, ...]
```

- **采样率（Sample Rate）**：每秒采多少次。常见值：
  - `16000 Hz`（16kHz）：语音任务常用（电话、语音识别）
  - `44100 Hz`（44.1kHz）：CD 音质
  - `48000 Hz`（48kHz）：专业音频

- **量化（Quantization）**：把每个采样点的振幅映射为整数。
  - 8 bit：256 个级别（电话音质）
  - 16 bit：65536 个级别（CD 音质，最常用）
  - 24 bit / 32 bit：专业录音

**奈奎斯特定理**：采样率必须 ≥ 信号最高频率的 2 倍，否则会失真（混叠）。人耳能听到 20Hz~20kHz，所以 CD 音质用 44.1kHz。

### 1.3 音频格式对比

| 格式 | 类型 | 特点 | AI 常用度 |
|------|------|------|----------|
| WAV | 无损 | 原始 PCM 数据，文件大 | ⭐⭐⭐⭐⭐ 训练首选 |
| FLAC | 无损压缩 | 体积约为 WAV 的 50~60% | ⭐⭐⭐ |
| MP3 | 有损压缩 | 体积小，质量有损失 | ⭐⭐ |
| OGG | 有损压缩 | 开源格式 | ⭐ |

**AI 训练推荐使用 WAV 格式**，因为：
1. 无需解码，直接读取为数值数组
2. 无损，保证数据质量
3. 所有框架（torchaudio、librosa、soundfile）都支持

### 1.4 从文件到张量

一段 1 秒钟、16kHz 采样率、单声道的音频，在计算机中就是一个**长度为 16000 的一维数组**：

```
音频文件 (.wav)
    ↓ 读取
张量 tensor([[-0.0658, -0.0709, -0.0753, ..., -0.0700, -0.0731, -0.0704]])
形状: torch.Size([1, 16000])
      ↑          ↑
    通道数    采样点数（1秒 × 16000Hz）
```

如果音频有多个声道（比如双声道立体声），第一维就是声道数（通常是 2）。

---

## 2. 通用的食用指南

在正式进入四大任务之前，先掌握音频处理的**通用工具链**。

### 2.1 核心库速查

| 库 | 用途 | 安装 |
|------|------|------|
| `torchaudio` | PyTorch 音频生态，数据集、变换、模型 | `pip install torchaudio` |
| `soundfile` | 音频文件读写（轻量、快速） | `pip install soundfile` |
| `librosa` | 音频特征提取（MFCC、频谱图等） | `pip install librosa` |
| `transformers` | HuggingFace 预训练音频模型 | `pip install transformers` |
| `datasets` | HuggingFace 音频数据集 | `pip install datasets` |

核心依赖：
```bash
pip install torchaudio soundfile transformers datasets
```

### 2.2 音频读写：soundfile vs torchaudio

两种主流方式读取音频文件：

In [ ]:
import soundfile as sf
import torchaudio
import torch

# ========== 方式 1: soundfile（推荐，轻量） ==========
wav, sample_rate = sf.read("audio.wav")
# wav 是 numpy 数组，shape = (采样点数,) 单声道 或 (采样点数, 通道数)
print(f"soundfile: shape={wav.shape}, sr={sample_rate}")

# 写入
sf.write("output.wav", wav, sample_rate)


# ========== 方式 2: torchaudio（PyTorch 生态） ==========
waveform, sr = torchaudio.load("audio.wav")
# waveform 是 tensor，shape = (通道数, 采样点数)
print(f"torchaudio: shape={waveform.shape}, sr={sr}")

# 写入
torchaudio.save("output.wav", waveform, sr)


# ========== 格式转换：多通道 → 单通道 ==========
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)  # 取平均
    print(f"转为单声道: {waveform.shape}")

### 2.3 音频可视化

最基本的可视化是画**波形图（Waveform）**——横轴时间、纵轴振幅。

In [ ]:
import matplotlib.pyplot as plt
import torchaudio

waveform, sr = torchaudio.load("audio.wav")

# 波形图
plt.figure(figsize=(14, 4))
plt.plot(waveform[0].numpy(), color="steelblue", linewidth=0.5)
plt.title(f"波形图 (采样率: {sr} Hz, 时长: {waveform.shape[1]/sr:.2f}s)")
plt.xlabel("采样点")
plt.ylabel("振幅")
plt.tight_layout()
plt.show()

### 2.4 频谱图（Spectrogram）

波形图只能看到时域信息。**频谱图**同时展示时间和频率：

```
波形图：  时间 → 振幅（只知道强弱）
频谱图：  时间 × 频率 → 能量（知道什么频率在什么时间出现）
```

**短时傅里叶变换（STFT）**：把音频切成短窗口（帧），对每帧做 FFT，得到频率分布。

**梅尔频谱图（Mel Spectrogram）**：在 STFT 基础上，用梅尔尺度（模拟人耳感知）压缩频率轴，是语音任务最常用的特征。

In [ ]:
import torchaudio.transforms as T
import matplotlib.pyplot as plt

waveform, sr = torchaudio.load("audio.wav")

# ========== STFT 频谱图 ==========
spectrogram = T.Spectrogram(n_fft=1024)
spec = spectrogram(waveform)  # shape: (1, freq_bins, time_frames)

plt.figure(figsize=(14, 4))
plt.imshow(spec[0].log2().numpy(), aspect="auto", origin="lower", cmap="magma")
plt.title("STFT 频谱图")
plt.xlabel("时间帧")
plt.ylabel("频率 bin")
plt.colorbar(label="log2(能量)")
plt.tight_layout()
plt.show()


# ========== 梅尔频谱图（更常用） ==========
mel_transform = T.MelSpectrogram(
    sample_rate=sr,
    n_fft=1024,
    hop_length=512,       # 帧移
    n_mels=128            # 梅尔滤波器数量
)
mel_spec = mel_transform(waveform)  # shape: (1, 128, time_frames)

plt.figure(figsize=(14, 5))
plt.imshow(mel_spec[0].log2().numpy(), aspect="auto", origin="lower", cmap="inferno")
plt.title(f"梅尔频谱图 (n_mels=128, sr={sr})")
plt.xlabel("时间帧")
plt.ylabel("梅尔频率 bin")
plt.colorbar(label="log2(能量)")
plt.tight_layout()
plt.show()

### 2.5 MFCC（梅尔频率倒谱系数）

MFCC 是传统语音任务中最经典的特征，它在梅尔频谱基础上再做一次**离散余弦变换（DCT）**，提取最能代表语音的系数。

```
音频 → STFT → 梅尔滤波器组 → 取对数 → DCT → MFCC（通常取 13~40 维）
```

MFCC 相比梅尔频谱图：维度更低、更紧凑，适合传统 ML 模型（GMM-HMM）。现代端到端模型（Wav2Vec2、Whisper）直接用原始波形或梅尔频谱图。

In [ ]:
import torchaudio.transforms as T
import matplotlib.pyplot as plt

waveform, sr = torchaudio.load("audio.wav")

mfcc_transform = T.MFCC(
    sample_rate=sr,
    n_mfcc=40,            # 提取 40 维 MFCC
    melkwargs={
        "n_fft": 1024,
        "hop_length": 512,
        "n_mels": 128
    }
)
mfcc = mfcc_transform(waveform)  # shape: (1, 40, time_frames)

plt.figure(figsize=(14, 4))
plt.imshow(mfcc[0].numpy(), aspect="auto", origin="lower", cmap="coolwarm")
plt.title(f"MFCC (40 维)")
plt.xlabel("时间帧")
plt.ylabel("MFCC 系数")
plt.colorbar()
plt.tight_layout()
plt.show()

### 2.6 关键参数速查表

| 参数 | 含义 | 常用值 |
|------|------|--------|
| `sample_rate` | 采样率 | 16000（语音）、44100（音乐） |
| `n_fft` | FFT 窗口大小 | 400、512、1024、2048 |
| `hop_length` | 帧移（窗口每次滑动的距离） | n_fft // 4（如 512 → hop=128） |
| `n_mels` | 梅尔滤波器数量 | 40、64、80、128 |
| `n_mfcc` | MFCC 系数数量 | 13、20、40 |

---

## 3. 音频分类（Audio Classification）

音频分类是语音任务中最直观的一类：给一段音频，输出它的类别标签。

**应用场景**：
- 语音指令识别（"前进"、"后退"、"停止"）
- 环境声音分类（鸟叫、汽车鸣笛、玻璃破碎）
- 音乐流派分类（流行、摇滚、古典）
- 情感识别（开心、愤怒、悲伤）

### 3.1 Pipeline 快速上手

HuggingFace 的 `pipeline` 是最简洁的方式，一行代码搞定：

In [ ]:
from transformers import pipeline
import torch

# 自动适配设备（CUDA / MPS / CPU）
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 加载语音分类 Pipeline
classifier = pipeline(
    task="audio-classification",
    model="MIT/ast-finetuned-speech-commands-v2",
    device=device
)

# 推理
result = classifier("audio.wav", top_k=3)

for r in result:
    print(f"  {r['label']:15s} → {r['score']:.4f}")

### 3.2 Pipeline 背后是什么？

Pipeline 封装了三个核心组件：

```
音频文件
  ↓
Feature Extractor（特征提取）  → 把原始波形转为模型需要的输入格式（梅尔频谱图）
  ↓
Model（模型推理）             → 对特征进行分类，输出 logits
  ↓
Post-processing（后处理）     → softmax → 取 top-k → 返回标签+置信度
```

In [ ]:
from transformers import pipeline

pipe = pipeline(
    task="audio-classification",
    model="MIT/ast-finetuned-speech-commands-v2"
)

# 查看 Pipeline 的内部结构
print(type(pipe.model))              # ASTForAudioClassification
print(type(pipe.feature_extractor))   # ASTFeatureExtractor（语音专属）
print(type(pipe.tokenizer))           # None（语音任务不需要 tokenizer）
print(type(pipe.image_processor))     # None（不是视觉任务）

# 模型架构
print(pipe.model.base_model)          # ASTModel（Audio Spectrogram Transformer）
print(pipe.model.classifier)          # 分类头

### 3.3 模型解剖：AST（Audio Spectrogram Transformer）

AST 是目前音频分类的主流模型，核心思想：

```
音频波形
  ↓ Mel Spectrogram
梅尔频谱图 (128×128)
  ↓ Patch Embedding（切成 16×16 小块）
  ↓ Transformer Encoder × 12 层
  ↓ [CLS] Token → 分类头
输出：类别概率
```

和 ViT（Vision Transformer）的思路完全一致：**把频谱图当图像来处理**。

### 3.4 不用 Pipeline：手动组装

理解 Pipeline 的内部原理后，可以手动组装，获得更细粒度的控制：

In [ ]:
import soundfile
from transformers import ASTFeatureExtractor, ASTForAudioClassification
import torch

# 1. 加载特征提取器
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-speech-commands-v2"
)

# 2. 加载模型
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-speech-commands-v2"
)

# 3. 读取音频
wav, sr = soundfile.read("audio.wav")

# 4. 特征提取（波形 → 梅尔频谱图张量）
inputs = feature_extractor(wav, sampling_rate=sr, return_tensors="pt")
print(f"输入特征形状: {inputs['input_values'].shape}")  # (1, 128, 128)

# 5. 推理
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # (1, num_labels)

# 6. 取预测结果
pred_id = logits.argmax(-1).item()
pred_label = model.config.id2label[pred_id]
print(f"预测: {pred_label}")

### 3.5 自定义训练（微调 AST）

用 HuggingFace `Trainer` 在自定义数据集上微调 AST，核心流程：

```
1. 准备数据集（SPEECHCOMMANDS / 自定义 WAV）
2. 自定义 Dataset 类，用 FeatureExtractor 处理每条音频
3. 构建 collate_fn 将 (x, y) 转为 {input_values, labels}
4. TrainingArguments + Trainer 训练
```

In [ ]:
from transformers import (
    ASTFeatureExtractor, ASTConfig, ASTForAudioClassification,
    TrainingArguments, Trainer
)
from torchaudio.datasets import SPEECHCOMMANDS
import torch, os, numpy as np

# ========== 1. 准备标签 ==========
ds_dir = "/path/to/your/dataset"
labels = sorted([d for d in os.listdir(ds_dir)
                  if os.path.isdir(os.path.join(ds_dir, d)) and not d.startswith("_")])
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

# ========== 2. 特征提取器 ==========
feature_extractor = ASTFeatureExtractor(
    feature_size=1, sampling_rate=16000,
    num_mel_bins=128, max_length=128,
    do_normalize=True, mean=-6.846, std=5.565
)

# ========== 3. 自定义 Dataset ==========
class ASTDataset(SPEECHCOMMANDS):
    def __init__(self, extractor, ds_path, subset, download=True):
        super().__init__(root=ds_path, download=download, subset=subset)
        self.extractor = extractor

    def __getitem__(self, idx):
        data, sr, label, _, _ = super().__getitem__(idx)
        feature = self.extractor(data[0], sr, return_tensors="pt")
        return feature["input_values"][0], torch.tensor(label2id[label])

ds_train = ASTDataset(feature_extractor, ds_dir, "training")
ds_valid = ASTDataset(feature_extractor, ds_dir, "validation")

# ========== 4. collate_fn ==========
def collate_fn(batch):
    x = torch.stack([item[0] for item in batch])
    y = torch.stack([item[1] for item in batch])
    return {"input_values": x, "labels": y}

# ========== 5. 构建模型 & 训练 ==========
config = ASTConfig(
    num_labels=len(labels), id2label=id2label, label2id=label2id,
    hidden_size=768, num_hidden_layers=12, num_attention_heads=12
)
model = ASTForAudioClassification(config)

args = TrainingArguments(
    output_dir="./trained_model",
    num_train_epochs=10, learning_rate=5e-5,
    per_device_eval_batch_size=24,
    eval_strategy="steps", eval_steps=100,
    save_strategy="epoch", save_only_model=True
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_valid,
    data_collator=collate_fn
)
trainer.train()

### 3.6 另一种思路：CNN 音频分类

不一定非得用 Transformer。对于小数据集或边缘设备，**轻量级 CNN** 往往更实用：

```
音频 → Mel Spectrogram (1, 128, T)
  ↓ Conv2d → BatchNorm → ReLU → MaxPool（重复 3~4 次）
  ↓ AdaptiveAvgPool → Flatten
  ↓ Linear → 分类输出
```

这种方案在实际项目（如语音控制系统）中非常常见，模型小、推理快、容易部署。

In [ ]:
import torch.nn as nn
from transformers import PreTrainedModel, PretrainedConfig

class SimpleCNNForAudio(PreTrainedModel):
    config_class = PretrainedConfig

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels

        layers = []
        in_ch = 1  # Mel 频谱图是单通道
        for out_ch in config.conv_channels:
            layers += [
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ]
            in_ch = out_ch
        layers.append(nn.AdaptiveAvgPool2d((4, 4)))
        self.conv = nn.Sequential(*layers)
        self.classifier = nn.Linear(config.conv_channels[-1] * 16, config.num_labels)
        self.post_init()

    def forward(self, input_values, labels=None, return_dict=True):
        x = self.conv(input_values).flatten(1)
        logits = self.classifier(x)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits} if return_dict else (loss, logits)

---

## 4. 音频到音频（Audio to Audio）

输入一段音频，输出另一段音频。核心任务：

| 任务 | 说明 | 典型模型 |
|------|------|----------|
| 语音增强 | 去除噪声，保留语音 | Sepformer, SpeechBrain |
| 源分离 | 从混合音频中分离各声源 | Conv-TasNet, Demucs |
| 超分辨率 | 提升音频采样率 | AudioSR |
| 声码器 | 频谱图 → 波形（TTS 的最后一步） | HiFi-GAN, WaveNet |

### 4.1 语音增强（Speech Enhancement）

目标：从含噪语音中恢复干净语音。

```
含噪语音 = 干净语音 + 噪声
      ↓
模型预测掩码（Mask）或直接预测干净语音
      ↓
输出：增强后的语音
```

In [ ]:
from transformers import pipeline

# HuggingFace Pipeline 支持 speech-enhancement
enhancer = pipeline(
    "automatic-speech-recognition",  # 或使用专门的增强模型
    model="speechbrain/mtl-mimic-english"  # 示例模型
)

# 实际中更常用 torchaudio 的频域操作
import torchaudio
import torchaudio.transforms as T

waveform, sr = torchaudio.load("noisy_audio.wav")

# 简单降噪：谱减法（Spectral Subtraction）
spec = T.Spectrogram(n_fft=1024)(waveform)
# 估计噪声谱（取前 0.5 秒的平均）
noise_spec = spec[:, :, :int(0.5 * sr / 512)].mean(dim=2, keepdim=True)
# 谱减法
clean_spec = (spec - noise_spec).clamp(min=0)
print(f"降噪后频谱形状: {clean_spec.shape}")

### 4.2 音频变换速查

torchaudio 提供了丰富的音频变换：

```python
import torchaudio.transforms as T

# 常用变换
T.Spectrogram(n_fft=1024)           # STFT 频谱图
T.MelSpectrogram(sample_rate=16000)  # 梅尔频谱图
T.MFCC(sample_rate=16000, n_mfcc=40) # MFCC 特征
T.Resample(orig_freq=44100, new_freq=16000)  # 重采样
T.Vol(gain=2.0)                      # 音量调节
T.TimeStretch()                      # 时间拉伸
T.PitchShift(sample_rate=16000, n_steps=4)  # 变调
T.FrequencyMasking(freq_mask_param=15)  # SpecAugment
T.TimeMasking(time_mask_param=35)        # SpecAugment
```

---

## 5. 语音识别（Automatic Speech Recognition）

语音识别（ASR）是语音任务中最重要的方向：**把语音转为文字**。

### 5.1 ASR 的发展脉络

```
传统方法 (2000s)                    端到端方法 (2020s)
┌─────────────────┐               ┌──────────────────────┐
│ 音频 → MFCC     │               │ 音频 → 原始波形       │
│   → GMM-HMM     │               │   → CNN/Transformer   │
│   → 解码器       │    ────→      │   → CTC / Attention   │
│   → 文字         │               │   → 文字              │
└─────────────────┘               └──────────────────────┘
  需要语言模型                      端到端，效果更好
  对齐复杂                          Wav2Vec2 / Whisper
```

### 5.2 Pipeline 快速上手

用 HuggingFace Pipeline 可以一行代码做 ASR：

In [ ]:
from transformers import pipeline

# Whisper 是目前最强的开源 ASR 模型
transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small"  # 也可用 whisper-base / medium / large
)

# 英文识别
result = transcriber("english_audio.wav")
print(result["text"])

# 中文识别（指定语言）
result = transcriber("chinese_audio.wav", generate_kwargs={"language": "zh"})
print(result["text"])

### 5.3 Whisper 架构解析

Whisper 是 OpenAI 开源的多语言 ASR 模型，采用 Encoder-Decoder 架构：

```
音频（30秒）
  ↓ Log-Mel Spectrogram (80 mel bins × 3000 frames)
  ↓
┌──────────────────────────┐
│   Encoder (Transformer)   │
│   6/12/24/32 层           │  ← 取决于模型大小
└──────────┬───────────────┘
           ↓
┌──────────────────────────┐
│   Decoder (Transformer)   │
│   自回归生成文本           │
└──────────┬───────────────┘
           ↓
      文字输出
```

Whisper 的训练数据来自 68 万小时的互联网音频，所以它天然支持：
- 多语言识别（99 种语言）
- 多任务（ASR、翻译、时间戳）
- 鲁棒性（对噪声、口音有较好容错）

### 5.4 Wav2Vec2：自监督预训练

Wav2Vec2 是 Meta（Facebook）提出的自监督 ASR 模型，核心创新：

```
原始波形
  ↓ CNN 特征提取器
  ↓ 量化模块
  ↓ Transformer Encoder
  ↓ 对比学习（自监督预训练）
  ↓ 微调（少量标注数据）
  ↓ CTC 解码 → 文字
```

优势：可以用**大量无标注音频**预训练，再用少量标注数据微调，特别适合低资源语言。

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch, soundfile

# 加载处理器和模型
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# 读取音频
wav, sr = soundfile.read("audio.wav")

# 特征提取
inputs = processor(wav, sampling_rate=sr, return_tensors="pt", padding=True)

# 推理
with torch.no_grad():
    logits = model(**inputs).logits

# CTC 解码
pred_ids = torch.argmax(logits, dim=-1)
text = processor.batch_decode(pred_ids)
print(f"识别结果: {text[0]}")

### 5.5 ASR 的关键概念

| 概念 | 说明 |
|------|------|
| **CTC (Connectionist Temporal Classification)** | 解决输入输出长度不一致的损失函数，无需逐帧对齐 |
| **Attention Decoder** | 自回归解码器，像 GPT 一样逐 token 生成 |
| **WER (Word Error Rate)** | ASR 评估指标，越低越好 |
| **VAD (Voice Activity Detection)** | 语音活动检测，过滤静音段 |
| **Language Model** | 语言模型，用于修正 ASR 输出（如 n-gram、LLM） |
| **Streaming ASR** | 流式识别，边说边出结果 |

### 5.6 语言识别（Language Identification）

语言识别是 ASR 的前置任务：先判断音频说的是什么语言，再选择对应模型识别。

核心流程与音频分类一致，只是标签变成了语言类别：

In [ ]:
from torchaudio.datasets import SPEECHCOMMANDS
import torch

# 加载数据集
ds_train = SPEECHCOMMANDS(root="/path/to/datasets", download=True, subset="training")
ds_valid = SPEECHCOMMANDS(root="/path/to/datasets", download=True, subset="validation")

# 查看数据结构
waveform, sr, label, speaker_id, wav_id = ds_train[0]
print(f"波形形状: {waveform.shape}")    # (1, 16000) → 单声道，1秒
print(f"采样率: {sr}")                 # 16000
print(f"标签: {label}")                # 'backward'
print(f"说话者: {speaker_id}")          # '0165e0e8'

# 保存为 WAV
torchaudio.save("output.wav", waveform, sr)

---

## 6. 语音合成（Text to Speech）

TTS 的目标是：**输入文字，输出自然语音**。

### 6.1 TTS 的发展脉络

```
拼接式 TTS (2000s)          参数式 TTS (2010s)          端到端 TTS (2020s)
┌─────────────┐          ┌──────────────┐          ┌─────────────────┐
│ 录音库切片    │          │ 文本 → 声学模型│          │ 文本 → Encoder   │
│ → 拼接还原    │   →      │ → 声学参数     │   →      │ → Attention      │
│ 机械不自然    │          │ → 声码器还原   │          │ → HiFi-GAN      │
└─────────────┘          └──────────────┘          │ → 自然语音       │
                                                    └─────────────────┘
                                                    Tacotron2 / VITS
```

### 6.2 Pipeline 快速上手

In [ ]:
from transformers import pipeline
import soundfile

# HuggingFace TTS Pipeline
synthesizer = pipeline("text-to-speech", model="facebook/mms-tts-eng")

result = synthesizer("Hello, this is a text to speech example.")

# result["audio"] 是 numpy 数组，result["sampling_rate"] 是采样率
soundfile.write("tts_output.wav", result["audio"], result["sampling_rate"])
print(f"已保存, 时长: {len(result['audio']) / result['sampling_rate']:.2f}s")

### 6.3 Tacotron2 + HiFi-GAN

torchaudio 内置了 Tacotron2 和 HiFi-GAN，可以本地运行：

In [ ]:
import torch
import torchaudio

# 加载预训练模型
bundle = torchaudio.pipelines.TACOTRON2_WAVERNN_CHAR_LJSPEECH
tacotron2 = bundle.get_model()
vocoder = bundle.get_vocoder()

# 文本转语音
text = "Hello, welcome to the world of speech synthesis."
with torch.inference_mode():
    lengths, predictions, _ = tacotron2.infer(text)
    waveforms = vocoder(predictions, lengths)

# 保存
torchaudio.save("tacotron2_output.wav", waveforms[0:1], bundle.sample_rate)
print(f"合成完成, 采样率: {bundle.sample_rate}")

### 6.4 TTS 的核心组件

```
文本
  ↓
┌──────────────┐
│  文本前端      │  拼音、韵律、停顿标记
└──────┬───────┘
       ↓
┌──────────────┐
│  声学模型      │  文本 → 梅尔频谱图（Tacotron2 / VITS）
└──────┬───────┘
       ↓
┌──────────────┐
│  声码器        │  频谱图 → 波形（HiFi-GAN / WaveNet / Griffin-Lim）
└──────┬───────┘
       ↓
    WAV 音频
```

| 组件 | 作用 | 典型模型 |
|------|------|----------|
| 文本前端 | 文本标准化、拼音转换、韵律标注 | g2p, pypinyin |
| 声学模型 | 文本 → 频谱图 | Tacotron2, VITS, FastSpeech2 |
| 声码器 | 频谱图 → 波形 | HiFi-GAN, WaveNet, Griffin-Lim |

---

## 7. 总结：四大任务全景图

```
                    ┌─────────────────────────────────────────┐
                    │             语音任务全景图               │
                    └─────────────────────────────────────────┘

    音频 ───→ ┌──────────────┐
              │ 音频分类       │ ──→ 类别标签
              │ (AST, CNN)    │     "backward", "stop"
              └──────────────┘

    音频 ───→ ┌──────────────┐
              │ 音频到音频     │ ──→ 增强/分离后的音频
              │ (Sepformer)   │     干净语音 / 分离音轨
              └──────────────┘

    音频 ───→ ┌──────────────┐
              │ 语音识别       │ ──→ 文字
              │ (Whisper)     │     "今天天气真好"
              └──────────────┘

    文字 ───→ ┌──────────────┐
              │ 语音合成       │ ──→ 音频
              │ (VITS)        │     WAV 文件
              └──────────────┘
```

### 核心要点

1. **特征工程是基础**：无论是哪种任务，第一步都是把波形转为梅尔频谱图或 MFCC
2. **Pipeline 是入口**：HuggingFace Pipeline 封装了特征提取、模型推理、后处理，适合快速验证
3. **Transformer 统治一切**：AST（分类）、Whisper（识别）、VITS（合成）都基于 Transformer
4. **WAV 是通用格式**：训练数据统一用 16kHz WAV，避免格式兼容问题
5. **端到端是趋势**：从 Wav2Vec2 到 Whisper，从 Tacotron2 到 VITS，都在减少人工设计的中间步骤

### 推荐学习路径

```
① 音频基础（本章 1~2 节）→ 理解采样、频谱图、特征提取
  ↓
② 音频分类（第 3 节）    → 掌握 Pipeline + AST 微调
  ↓
③ 语音识别（第 5 节）    → Whisper / Wav2Vec2 实战
  ↓
④ 语音合成（第 6 节）    → Tacotron2 + HiFi-GAN
  ↓
⑤ 实战项目              → 语音控制系统（项目 07）
```

---

## 参考资料

- [HuggingFace Audio Course](https://huggingface.co/learn/audio-course)
- [torchaudio 官方文档](https://pytorch.org/audio/stable/)
- [Whisper 论文](https://arxiv.org/abs/2212.04356)
- [Wav2Vec2 论文](https://arxiv.org/abs/2006.11477)
- [AST 论文](https://arxiv.org/abs/2104.01778)
- [HiFi-GAN 论文](https://arxiv.org/abs/2010.05646)